# slide2pptx: train the element detector (YOLO)

**Runtime → Change runtime type → T4 GPU.** Needs the synthetic dataset made by
`colab_setup.ipynb` in Google Drive. Runs are saved to Drive, so a disconnect only
loses the current epoch: re-run cells 1–3, then the **resume** cell.

In [ ]:
# 1. Settings
REPO_URL  = "https://github.com/C00p1r/slide2pptx.git"
DRIVE_DIR = "/content/drive/MyDrive/slide2pptx"
DATA      = f"{DRIVE_DIR}/data/synth"          # from colab_setup.ipynb
MODEL     = "yolo11s.pt"                         # yolo11n.pt is ~2x faster, yolo11m.pt more accurate
EPOCHS    = 40
IMGSZ     = 1280                                 # small text/icons need large images
BATCH     = 8                                    # T4 (15 GB) at 1280 with yolo11s; lower if out of memory
RUN_NAME  = "y11s_1280_sim2real"
RUNS      = f"{DRIVE_DIR}/runs/detect"

In [ ]:
# 2. Mount Drive, clone/update the repo, install
from google.colab import drive
drive.mount('/content/drive')
import os
if not os.path.exists('/content/slide2pptx'):
    !git clone {REPO_URL} /content/slide2pptx
else:
    !git -C /content/slide2pptx pull
%cd /content/slide2pptx
!pip install -q -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 3. Copy the dataset to local disk (reading thousands of files from Drive is very slow)
LOCAL = "/content/synth"
if not os.path.exists(LOCAL):
    !cp -r {DATA} {LOCAL}
!for s in train val test; do echo $s $(ls {LOCAL}/images/$s | wc -l); done

Quick check (optional): 1 epoch on 10% of the data, to make sure everything runs before the long job.

In [ ]:
!python -m detector.train --data {LOCAL}/data.yaml --model {MODEL} --epochs 1 --fraction 0.1 \
    --imgsz {IMGSZ} --batch {BATCH} --project /content/runs --name smoke

In [ ]:
# 4. Train (yolo11s, 5k slides, 1280: roughly 3-5 min per epoch on a T4, so 40 epochs may need one resume)
!python -m detector.train --data {LOCAL}/data.yaml --model {MODEL} --epochs {EPOCHS} \
    --imgsz {IMGSZ} --batch {BATCH} --project {RUNS} --name {RUN_NAME}

In [ ]:
# 4b. Resume after a disconnect (re-run cells 1-3 first)
!python -m detector.train --data {LOCAL}/data.yaml --project {RUNS} --name {RUN_NAME} --resume

In [ ]:
# 5. Per-class mAP on the synthetic test split -> Drive/results/detector/synth_test.md
W = f"{RUNS}/{RUN_NAME}/weights/best.pt"
!python -m detector.evaluate {W} --data {LOCAL}/data.yaml --split test --imgsz {IMGSZ} \
    --out {DRIVE_DIR}/results/detector --tag {RUN_NAME}_synth

## Real NotebookLM slides
Put the slide images in `DRIVE_DIR/data/real/images` (see `tools/extract_slides.py`).
Until they are hand-labelled (`docs/labelling.md`), we can only look at predictions.

In [ ]:
REAL = f"{DRIVE_DIR}/data/real"
!python -m detector.predict {W} {REAL}/images --out {DRIVE_DIR}/results/pred_real --imgsz {IMGSZ}
import glob
from IPython.display import Image, display
for p in sorted(glob.glob(f"{DRIVE_DIR}/results/pred_real/*.jpg"))[:6]:
    display(Image(p, width=900))

In [ ]:
# Pre-labels for CVAT (docs/labelling.md, step 2)
!python tools/real_labels.py prelabel {W} --images {REAL}/images --out {REAL}/cvat_prelabels.zip

In [ ]:
# After labelling: build the real test set from the CVAT export and get real mAP
EXPORT = f"{REAL}/cvat_export.zip"     # upload your CVAT 'YOLO 1.1' export here
!python tools/real_labels.py build {EXPORT} --images {REAL}/images --out /content/real_yolo
!python -m detector.evaluate {W} --data /content/real_yolo/data.yaml --imgsz {IMGSZ} \
    --out {DRIVE_DIR}/results/detector --tag {RUN_NAME}_real

## Ablation for the report: does sim-to-real augmentation help?
Train the same model without the extra JPEG/blur/downscale augmentation and compare
real-slide mAP. This is the project's key result (sim-to-real gap).

In [ ]:
!python -m detector.train --data {LOCAL}/data.yaml --model {MODEL} --epochs {EPOCHS} \
    --imgsz {IMGSZ} --batch {BATCH} --project {RUNS} --name {RUN_NAME}_noaug --no-sim2real

## Experiment: which synthetic training set transfers best to real slides?
Three training sets of the **same size**, made with `colab_setup.ipynb` and
`python -m slidegen.generate --random-ratio R` (needs the generator PR with that flag):

| set | `--random-ratio` | folder |
|---|---|---|
| templates only | 0 | `synth_tpl` |
| random layouts only | 1 | `synth_rand` |
| 80/20 mix | 0.2 | `synth_mix` |

Keep everything else equal (model, epochs, imgsz, seed, sim2real on), and compare on the
**real** slides. Each set is also scored on its own synthetic test split for reference.
Four full runs do not fit in one free Colab day: run one set per session (resume works),
or use `EPOCHS = 30` for all three.

In [ ]:
SETS = {"tpl": f"{DRIVE_DIR}/data/synth_tpl", "rand": f"{DRIVE_DIR}/data/synth_rand", "mix": f"{DRIVE_DIR}/data/synth_mix"}
SET = "tpl"                         # run once per set: "tpl", then "rand", then "mix"
local = f"/content/{SET}"
if not os.path.exists(local):
    !cp -r {SETS[SET]} {local}
!python -m detector.train --data {local}/data.yaml --model {MODEL} --epochs {EPOCHS} \
    --imgsz {IMGSZ} --batch {BATCH} --project {RUNS} --name set_{SET}
!python -m detector.evaluate {RUNS}/set_{SET}/weights/best.pt --data {local}/data.yaml --split test \
    --imgsz {IMGSZ} --out {DRIVE_DIR}/results/detector --tag set_{SET}_synth

In [ ]:
# After all three runs and the real labels (/content/real_yolo from the cell above):
for s in ["tpl", "rand", "mix"]:
    !python -m detector.evaluate {RUNS}/set_{s}/weights/best.pt --data /content/real_yolo/data.yaml \
        --imgsz {IMGSZ} --out {DRIVE_DIR}/results/detector --tag set_{s}_real
R = f"{DRIVE_DIR}/results/detector"
!python -m detector.compare {R}/set_tpl_real.json {R}/set_rand_real.json {R}/set_mix_real.json --out {R}/compare_sets_real.md